In [ ]:
pip install vllm wget pandas openai pyarrow fastparquet openpyxl
python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen3-30B-A3B-Instruct-2507 --dtype bfloat16 --max-model-len 16384 --gpu-memory-utilization 0.90

In [2]:
import os
os.environ.setdefault("VLLM_BASE_URL", "http://127.0.0.1:8000/v1")
os.environ.setdefault("VLLM_API_KEY", "vx")  # qualquer string não vazia
os.environ.setdefault("VLLM_MODEL", "Qwen/Qwen3-30B-A3B-Instruct-2507")

# valor de contexto que você configurou no servidor (acima usamos 16384)
os.environ.setdefault("VLLM_MAX_MODEL_LEN", "16384")

'16384'

In [3]:
# ============================================
# 1) vllm_client.py (sincrono, sem asyncio)
# ============================================
CODE = r'''
import os, re, json, time, math
from typing import List, Dict, Any, Optional
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

from openai import OpenAI
from transformers import AutoTokenizer

VLLM_BASE_URL = os.environ.get("VLLM_BASE_URL", "http://127.0.0.1:8000/v1")
VLLM_API_KEY  = os.environ.get("VLLM_API_KEY", "vx")
VLLM_MODEL_ID = os.environ.get("VLLM_MODEL", "Qwen/Qwen3-30B-A3B-Instruct-2507")
MAX_MODEL_LEN = int(os.environ.get("VLLM_MAX_MODEL_LEN", "16384"))

# Cliente OpenAI-compatível (vLLM)
client = OpenAI(base_url=VLLM_BASE_URL, api_key=VLLM_API_KEY)

# Tokenizer apenas para contagem/split (não carrega pesos do modelo)
tokenizer = AutoTokenizer.from_pretrained(VLLM_MODEL_ID, trust_remote_code=True)

def healthcheck() -> bool:
    try:
        _ = client.models.list()
        return True
    except Exception as e:
        print("vLLM offline?", e)
        return False

def build_messages(system_prompt: str, user_content: str):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_content},
    ]

def _best_effort_json(text: str) -> Optional[Dict[str, Any]]:
    # remove cercas ```json ... ```
    t = text.strip()
    if t.startswith("```"):
        t = re.sub(r"^```(?:json)?\s*|\s*```$", "", t, flags=re.I|re.S)
    # tenta JSON direto
    try:
        return json.loads(t)
    except Exception:
        pass
    # extrai primeiro objeto {...}
    m = re.search(r"\{(?:[^{}]|(?R))*\}", t, flags=re.S)  # recursivo simples
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            return None
    return None

def score_chunks_threaded(system_prompt: str,
                          user_template: str,
                          chunks: List[str],
                          max_tokens_out: int = 200,
                          workers: int = 8,
                          force_json: bool = True) -> List[Dict[str, Any]]:
    def _work(ch: str) -> Dict[str, Any]:
        user_msg = user_template.format(text=ch)
        try:
            out = chat_once(system_prompt, user_msg,
                            max_tokens=max_tokens_out,
                            temperature=0.2, seed=42,
                            force_json=force_json)
        except Exception as e:
            return {"error": f"llm_call_failed: {e}"}
        pj = _best_effort_json(out)
        return pj if pj is not None else {"error": "no_json", "raw_sample": out[:160]}

def _count_tokens(txt: str) -> int:
    return len(tokenizer.encode(txt))

def suggest_chunk_tokens(system_prompt: str,
                         user_template: str,
                         reserve_gen: int = 128,
                         safety_margin: int = 128) -> int:
    """Calcula o tamanho máximo de chunk dado o contexto do servidor."""
    prompt_tokens = _count_tokens(system_prompt) + _count_tokens(user_template.format(text=""))
    budget = MAX_MODEL_LEN - prompt_tokens - reserve_gen - safety_margin
    # Limite superior prudente para 16k de contexto:
    return max(1024, min(budget, 12000))

def split_text_by_tokens(text: str, max_tokens: int, overlap: int = 64) -> List[str]:
    ids = tokenizer.encode(text)
    out = []
    i = 0
    while i < len(ids):
        j = min(i + max_tokens, len(ids))
        out.append(tokenizer.decode(ids[i:j]))
        if j == len(ids): break
        i = max(0, j - overlap)
    return out

def chat_once(system_prompt: str,
              user_content: str,
              max_tokens: int = 200,
              temperature: float = 0.2,
              seed: int = 42,
              force_json: bool = True) -> str:
    kwargs = dict(
        model=VLLM_MODEL_ID,
        messages=build_messages(system_prompt, user_content),
        temperature=temperature,
        max_tokens=max_tokens,
        seed=seed,
    )
    if force_json:
        kwargs["response_format"] = {"type": "json_object"}
    try:
        resp = client.chat.completions.create(**kwargs)
    except Exception:
        # Fallback p/ builds vLLM sem response_format
        kwargs.pop("response_format", None)
        resp = client.chat.completions.create(**kwargs)
    return resp.choices[0].message.content

def score_chunks_threaded(system_prompt: str,
                          user_template: str,
                          chunks: List[str],
                          max_tokens_out: int = 200,
                          workers: int = 8,
                          force_json: bool = True) -> List[Dict[str, Any]]:
    """Dispara requisições por chunk em paralelo (threads), mantendo API síncrona."""
    def _work(ch: str) -> Dict[str, Any]:
        user_msg = user_template.format(text=ch)
        out = chat_once(system_prompt, user_msg,
                        max_tokens=max_tokens_out,
                        temperature=0.2, seed=42,
                        force_json=force_json)
        pj = _best_effort_json(out)
        return pj if pj is not None else {"raw": out}

    results = [None] * len(chunks)
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futures = {ex.submit(_work, ch): idx for idx, ch in enumerate(chunks)}
        for fut in as_completed(futures):
            idx = futures[fut]
            try:
                results[idx] = fut.result()
            except Exception as e:
                results[idx] = {"error": str(e)}
    return results

# --- Agregadores ---
DEFAULT_AXIS_ORDER = [
    "People-Centrism", "Anti-Elitism", "MoralDichotomy",
    "PopularSovereigntyClaim", "ExclusionaryRhetoric"
]

def aggregate_axis_scores(items: List[Dict[str, Any]],
                          axis_order: Optional[List[str]] = None) -> Dict[str, Any]:
    """Agrega lista de JSONs no formato {"axisScores":[{"axisName":..,"score":..},...]}."""
    # Determinar ordem a partir do primeiro item válido ou usar default
    if axis_order is None:
        axis_order = None
        for it in items:
            if isinstance(it, dict) and isinstance(it.get("axisScores"), list) and it["axisScores"]:
                names = []
                for e in it["axisScores"]:
                    if isinstance(e, dict) and "axisName" in e:
                        names.append(e["axisName"])
                if names:
                    axis_order = names
                    break
        if axis_order is None:
            axis_order = DEFAULT_AXIS_ORDER

    sums = defaultdict(float)
    cnts = defaultdict(int)

    for it in items:
        if not isinstance(it, dict): 
            continue
        arr = it.get("axisScores")
        if not isinstance(arr, list):
            continue
        for e in arr:
            if not isinstance(e, dict): 
                continue
            name = e.get("axisName")
            val = e.get("score", None)
            if name in axis_order and isinstance(val, (int, float)):
                sums[name] += float(val)
                cnts[name] += 1

    out = []
    for name in axis_order:
        if cnts[name] > 0:
            mean = sums[name] / cnts[name]
            out.append({"axisName": name, "score": round(mean, 2)})
        else:
            out.append({"axisName": name, "score": None})
    return {"axisScores": out}

def aggregate_generic(items: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Fallback simples quando não há 'axisScores'."""
    # média de numéricos no topo; strings descartadas
    acc = defaultdict(list)
    for it in items:
        if not isinstance(it, dict): 
            continue
        for k, v in it.items():
            if isinstance(v, (int, float)):
                acc[k].append(float(v))
    out = {}
    for k, arr in acc.items():
        if arr:
            out[k] = round(sum(arr)/len(arr), 4)
    return out

def aggregate_items(items: List[Dict[str, Any]]) -> Dict[str, Any]:
    has_axis = any(isinstance(it, dict) and "axisScores" in it for it in items)
    if has_axis:
        return aggregate_axis_scores(items)

    # coleta erros para diagnósticos
    errs = [it.get("error") for it in items if isinstance(it, dict) and "error" in it]
    if len(errs) == len(items) and len(items) > 0:
        # todos falharam
        from collections import Counter
        common = Counter(errs).most_common(1)[0][0]
        return {"error": f"all_chunks_failed:{common}"}

    # fallback numérico simples
    return aggregate_generic(items)

def process_long_text(system_prompt: str,
                      user_template: str,
                      long_text: str,
                      overlap_tokens: int = 64,
                      max_tokens_out: int = 200,
                      workers: int = 8,
                      reserve_gen: int = 128,
                      safety_margin: int = 128) -> Dict[str, Any]:
    """Split → chamadas paralelas síncronas → agregação."""
    chunk_tok = suggest_chunk_tokens(system_prompt, user_template,
                                     reserve_gen=reserve_gen, safety_margin=safety_margin)
    chunks = split_text_by_tokens(long_text, max_tokens=chunk_tok, overlap=overlap_tokens)
    t0 = time.time()
    per_chunk = score_chunks_threaded(system_prompt, user_template, chunks,
                                      max_tokens_out=max_tokens_out,
                                      workers=workers, force_json=True)
    agg = aggregate_items([p for p in per_chunk if isinstance(p, dict)])
    return {
        "n_chunks": len(chunks),
        "chunk_tokens": chunk_tok,
        "latency_s": round(time.time() - t0, 2),
        "per_chunk": per_chunk,
        "aggregate": agg
    }

def to_json_str(obj: Any) -> str:
    try:
        return json.dumps(obj, ensure_ascii=False, separators=(",", ":"))
    except Exception as e:
        return json.dumps({"error": f"dump_failed: {e}"}, ensure_ascii=False)
'''
with open("vllm_client.py", "w", encoding="utf-8") as f:
    f.write(CODE)

print("vllm_client.py criado/atualizado.")


vllm_client.py criado/atualizado.


In [4]:
import wget
wget.download('https://github.com/andrepyles/teste/releases/download/teste2/DiscourseDB_v1.0.parquet')

'DiscourseDB_v1.0.parquet'

In [10]:
import pandas as pd
from tqdm import tqdm
from vllm_client import (
    healthcheck, process_long_text, to_json_str
)

assert healthcheck(), "Servidor vLLM não encontrado. Suba o api_server e confira VLLM_BASE_URL."

# Arquivo/parâmetros — ajuste conforme seu caso
INPUT_PARQUET  = "./DiscourseDB_v1.0.parquet"
OUTPUT_PARQUET = "./popin.parquet"
TEXT_COL       = "DISCOURSE"            # coluna com o discurso
OUTPUT_COL     = "POPIN_JSON"    # coluna a ser preenchida com o JSON final

SYSTEM_PROMPT = """
<NO-THINK>Do NOT produce <think> or chain-of-thought. Output ONLY the final JSON.</NO-THINK>
You are a political science research assistant specialized in comparative populism.
Analyze the political speech using the 5-dimensional framework below and output ONLY VALID JSON (no spaces/newlines).

CONCEPTUAL GUIDE (for you; do NOT include in output):
People-Centrism = unified & virtuous “people”, paramount will. (Mudde, 2004)
Anti-Elitism = “elites” as corrupt/self-serving/anti-people. (Weyland, 2017)
MoralDichotomy = “good people” vs “evil elites/outsiders”. (Mudde & Kaltwasser, 2017)
PopularSovereigntyClaim = leader claims to embody/override institutions in the name of the people. (Pappas, 2019)
ExclusionaryRhetoric = defines “the people” by excluding out-groups. (Moffitt, 2016)

LANGUAGE + CHUNK SAFETY:
- The text may be Portuguese/Spanish/English; score in a language-agnostic way.
- Treat the provided text as a self-contained excerpt (a chunk). Do not assume context beyond it. Score only what is present in THIS input.

AXIS NAME LOCK (exact strings, exact order):
1) People-Centrism
2) Anti-Elitism
3) MoralDichotomy
4) PopularSovereigntyClaim
5) ExclusionaryRhetoric

CUE INTERPRETATION (nuance rules):
- Count explicit, normative claims, not mere mentions. E.g., generic “o povo brasileiro” without normative force ≠ People-Centrism.
- Weight direct assertions > metaphors > implications. Weight repeated, prominent, and varied cues higher than isolated or offhand cues.
- Penalize conflations: criticizing policy competence or “inefficiency” alone ≠ Anti-Elitism without moralized “elite vs povo”.
- Institutional deference (“respeito às instituições”, “checks and balances”) lowers PopularSovereigntyClaim unless overridden by claims to embody the will of the people.
- ExclusionaryRhetoric requires boundary-drawing that narrows “the people” (e.g., “verdadeiros cidadãos”, “inimigos internos/externos”), not general references to crime or corruption without group exclusion.
- Do not reward tone/aggressiveness if the required cues are absent.

CONTINUOUS SCORING:
- Each axis is a continuous intensity index in [0.00,100.00]. Use 0.01 resolution and ALWAYS format with two decimals (e.g., 63.27, 41.58). Avoid round numbers by default (i.e., prefer non-.00/.50 unless the text truly supports exact thresholds).
- Soft floor: if ANY qualifying cue appears for an axis, assign a small non-zero in 1.00–9.99 with non-round decimals (e.g., 3.14, 7.83). Use 0.00 ONLY if no qualifying cues are present for that axis.
- Moderate presence: ≈25.00–60.00; strong: ≈60.01–84.99; extreme: ≈85.00–100.00.
- Length normalization: do not inflate scores because the excerpt is long; score by intensity and salience of cues, not token count.
- Cross-axis independence: score each axis on its own evidence; correlated movement is allowed but not forced.

NOT-SPEECH HANDLING:
- If the text is not a political speech/excerpt (e.g., news article, table, code, contract) set all five scores to null.

OUTPUT SCHEMA (STRICT JSON, MINIFIED):
{"axisScores":[
{"axisName":"People-Centrism","score":0-100(float,2 decimals|null)},
{"axisName":"Anti-Elitism","score":0-100(float,2 decimals|null)},
{"axisName":"MoralDichotomy","score":0-100(float,2 decimals|null)},
{"axisName":"PopularSovereigntyClaim","score":0-100(float,2 decimals|null)},
{"axisName":"ExclusionaryRhetoric","score":0-100(float,2 decimals|null)}]
}

FEW-SHOT EXAMPLE (for format and non-zero anchoring; do NOT copy the scores):
{"axisScores":[
{"axisName":"People-Centrism","score":62.73},
{"axisName":"Anti-Elitism","score":71.28},
{"axisName":"MoralDichotomy","score":58.41},
{"axisName":"PopularSovereigntyClaim","score":44.36},
{"axisName":"ExclusionaryRhetoric","score":15.92}]}

NOT_SPEECH EXAMPLE (format only):
{"axisScores":[
{"axisName":"People-Centrism","score":null},
{"axisName":"Anti-Elitism","score":null},
{"axisName":"MoralDichotomy","score":null},
{"axisName":"PopularSovereigntyClaim","score":null},
{"axisName":"ExclusionaryRhetoric","score":null}]}

RULES:
- Use ONLY the provided text. Keep the exact axis order/names. Output only the minimal schema above.
- Sanity: if ANY qualifying populist cue exists in the text, at least one axis MUST be >0.00.
- If text has <50 words, return exactly: {"error":"Speech too short (<50 words)"} (minified).
- Do NOT include quotes/evidence or any extra fields. Output only the final JSON.
Begin when the speech text is provided. Output ONLY the final JSON.
"""

USER_TEMPLATE = """Analyse this text and output ONLY the final JSON:
{text}"""

df = pd.read_parquet(INPUT_PARQUET).sample(1000)

if TEXT_COL not in df.columns:
    raise ValueError(f"Coluna '{TEXT_COL}' não encontrada no parquet.")

# Processamento linha a linha (síncrono), com barra de progresso
outputs = []
for txt in tqdm(df[TEXT_COL].astype(str).tolist(), desc="Processando discursos com vLLM"):
    try:
        res = process_long_text(
            system_prompt=SYSTEM_PROMPT,
            user_template=USER_TEMPLATE,
            long_text=txt,
            overlap_tokens=192,      # 32–64 é um bom valor; aumente só se notar perda na borda
            max_tokens_out=300,     # sua saída é um JSON curto; 96–200 é suficiente
            workers=8               # H200: 8–12 costuma ir bem; L40S: 4–8
        )
        final_json = res["aggregate"]
        outputs.append(to_json_str(final_json if final_json else {"error":"empty_aggregate"}))
    except Exception as e:
        outputs.append(to_json_str({"error": f"processing_failed: {e}"}))

# Escreve a coluna com o JSON minificado
df[OUTPUT_COL] = outputs

# Salva em parquet (mantendo as demais colunas)
df.to_parquet(OUTPUT_PARQUET, index=False)

print("Concluído.")
print("Exemplo do JSON gerado na primeira linha:")
print(df[OUTPUT_COL].iloc[0])

Processando discursos com vLLM: 100%|██████████| 44719/44719 [8:42:54<00:00,  1.43it/s]   


Concluído.
Exemplo do JSON gerado na primeira linha:
{"axisScores":[{"axisName":"People-Centrism","score":78.42},{"axisName":"Anti-Elitism","score":65.31},{"axisName":"MoralDichotomy","score":72.15},{"axisName":"PopularSovereigntyClaim","score":58.67},{"axisName":"ExclusionaryRhetoric","score":23.89}]}


In [11]:
df.drop(columns='DISCOURSE').to_excel('popin.xlsx')